about the two images:-

distance between white box and camera is 618mm

distance between matches box and metal box is 212mm

distance between metal box and camera is 410mm

distance between matches box and camera is 460mm

disparity of camera in x direction (shift right) is 25mm

height of camera from ground is 170mm

object -- matches box -- white box -- metal box

width  --  52mm       --   175mm   --  82mm

height --  70mm       --   55mm    --  127mm

matches box width is 148px and height is 203px

Z = (f.B)/d

where Z is distance from camera , f focal length depend on direction, B baseline width (width in real),

d width in camera by pixel

In [1]:
import cv2
import numpy as np

In [7]:
# 1. Load your two images in grayscale
# img1 is before moving, img2 is after moving 1cm to the right
img1 = cv2.imread('images/original.jpeg', cv2.IMREAD_GRAYSCALE)
img2 = cv2.imread('images/shift_right.jpeg', cv2.IMREAD_GRAYSCALE)

if img1 is None:
    print("Error: Could not load 'photo_left.jpg'. Check the file name and path!")
if img2 is None:
    print("Error: Could not load 'photo_right.jpg'. Check the file name and path!")

if img1 is not None and img2 is not None:
    print(f"Images loaded successfully! Image 1 resolution: {img1.shape}")

# 2. Initialize the ORB detector
orb = cv2.ORB_create(nfeatures=1000)

# 3. Find keypoints and compute descriptors for both images
kp1, des1 = orb.detectAndCompute(img1, None)
kp2, des2 = orb.detectAndCompute(img2, None)

# 4. Match descriptors using Brute-Force Matcher
bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
matches = bf.match(des1, des2)

# Sort matches by distance (best matches first)
matches = sorted(matches, key=lambda x: x.distance)

print(f"Total keypoints in Image 1: {len(kp1)}")
print(f"Total keypoints in Image 2: {len(kp2)}")
print(f"Total raw matches found: {len(matches)}")

# 5. Extract coordinates and calculate horizontal disparity (d)

# Define your camera intrinsic parameters (Replace these with your actual calibration values!)
# If you haven't calibrated yet, you can estimate fx as roughly equal to image width (1600) for now
fx = 1309.231  
fy = 1334.0
cx = 800.0   # Roughly half of width (1600 / 2)
cy = 602.0   # Roughly half of height (1204 / 2)

B = 2.5      # Baseline is 1 cm (keeping units in centimeters)

valid_points = []

for match in matches:
    x1, y1 = kp1[match.queryIdx].pt
    x2, y2 = kp2[match.trainIdx].pt
    
    disparity_x = x1 - x2
    disparity_y = y1 - y2
    
    # 1. Loosen the filter to 45 pixels to accommodate manual box movement
    if abs(disparity_y) < 45.0 and disparity_x > 0:
        
        # 2. Calculate Depth (Z) using fx and baseline
        # Using disparity_x because the camera moved horizontally
        Z = (fx * B) / disparity_x
        
        # 3. Calculate X and Y coordinates relative to the optical center
        X = ((x1 - cx) * Z) / fx
        Y = ((y1 - cy) * Z) / fy
        
        valid_points.append({
            "pixel_left": (x1, y1),
            "disparity": disparity_x,
            "3D_coord": (X, Y, Z)
        })

# Print out your final 3D Cloud Points!
print(f"Successfully tracked {len(valid_points)} features.")
print("\n--- Generated 3D Cloud Points (First 5) ---")
for i, pt in enumerate(valid_points[:5]):
    X, Y, Z = pt["3D_coord"]
    print(f"Point {i} at Pixel ({pt['pixel_left'][0]:.1f}, {pt['pixel_left'][1]:.1f}) -> 3D Space: X={X:.2f}cm, Y={Y:.2f}cm, Z={Z:.2f}cm")

Images loaded successfully! Image 1 resolution: (1600, 1204)
Total keypoints in Image 1: 1000
Total keypoints in Image 2: 911
Total raw matches found: 374
Successfully tracked 292 features.

--- Generated 3D Cloud Points (First 5) ---
Point 0 at Pixel (288.0, 1118.9) -> 3D Space: X=-21.68cm, Y=21.48cm, Z=55.44cm
Point 1 at Pixel (288.2, 1119.7) -> 3D Space: X=-21.28cm, Y=21.12cm, Z=54.43cm
Point 2 at Pixel (882.0, 440.0) -> 3D Space: X=8.54cm, Y=-16.56cm, Z=136.38cm
Point 3 at Pixel (630.7, 1474.0) -> 3D Space: X=-5.44cm, Y=27.51cm, Z=42.09cm
Point 4 at Pixel (912.0, 250.0) -> 3D Space: X=8.24cm, Y=-25.40cm, Z=96.27cm


In [8]:
# Draw the top 50 best matches
img_matches = cv2.drawMatches(img1, kp1, img2, kp2, matches[:50], None, flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)

# Display the image window
cv2.imshow("Tracked Features Across Movement", img_matches)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [9]:
# --- Save the generated 3D cloud points to a text file ---
output_filename = "cloud_points.txt"

with open(output_filename, "w") as f:
    for pt in valid_points:
        X, Y, Z = pt["3D_coord"]
        # Save formatted as x,y,z on a new line
        f.write(f"{X:.4f},{Y:.4f},{Z:.4f}\n")

print(f"\nSuccessfully saved {len(valid_points)} cloud points to '{output_filename}'!")


Successfully saved 292 cloud points to 'cloud_points.txt'!
